In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent / "src"))

import torch
from torch.utils.data import DataLoader
from multiprocessing import Manager
import matplotlib.pyplot as plt

from config import read_model_config
from data import GDPHMMSlicesDataset, SharedCache
from model_sampler import build_sampler
from models import MambaUnet
from utils.mp import detect_device
from utils.checkpointing import load_checkpoint

In [3]:
model_name = '021-llr-00001'
epoch = 140
data_split = 'train'

In [ ]:
model_config = read_model_config(model_name)
device = detect_device(None)

manager = Manager()
dummy_cache = SharedCache(manager)
dataset = GDPHMMSlicesDataset(
  dataset_name=model_config['data']['dataset_name'],
  split=data_split,
  img_size=model_config['data']['img_size'],
  cache=dummy_cache,
)

dataloader = DataLoader(
  dataset,
  batch_size=1,
  shuffle=False,
)

ema_model = MambaUnet(
  img_size=model_config['data']['img_size'],
  in_struc_channels=dataset.n_structures
).to(device)

checkpoint = load_checkpoint(model_name, epoch)
ema_model.load_state_dict(checkpoint['ema_model_state_dict'])

sampler = build_sampler(model_config, ema_model, ema_model)

In [ ]:
structure, target, info = next(iter(dataloader))
structure = structure.to(device)
target = target.to(device)

x_T = torch.randn_like(target, device=device)
pred = sampler(x_T, structure)
print(pred.shape)

plt.figure(figsize=(8, 8))
plt.imshow(pred[0, 0].cpu().numpy(), cmap='gray')